# Experiment Comparison
Loads results from all subdirectories under `results/` and compares them across key metrics.

In [1]:
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RESULTS_DIR = Path("results")

# ── Load all experiments ───────────────────────────────────────────────────────
experiments = {}
for exp_dir in sorted(RESULTS_DIR.iterdir()):
    if not exp_dir.is_dir():
        continue
    metrics_file = exp_dir / "metrics.json"
    losses_file  = exp_dir / "losses.json"
    if not metrics_file.exists():
        continue
    entry = {"name": exp_dir.name}
    with open(metrics_file) as f:
        entry["metrics"] = json.load(f)
    if losses_file.exists():
        with open(losses_file) as f:
            entry["losses"] = json.load(f)

    # protein metrics (first protein found)
    protein_dirs = list((exp_dir / "protein_evaluation").glob("*/metrics.json")) if (exp_dir / "protein_evaluation").exists() else []
    if protein_dirs:
        with open(protein_dirs[0]) as f:
            entry["protein"] = json.load(f)

    experiments[exp_dir.name] = entry

print(f"Loaded {len(experiments)} experiments:", list(experiments.keys()))


Loaded 19 experiments: ['baseline', 'best_combo', 'best_combo_long', 'deep_architecture', 'gelu_activation', 'gelu_shallow', 'high_dropout', 'high_neg_multiplier', 'higher_aug_temperature', 'longer_training', 'low_lr', 'no_augmentation', 'shallow_architecture', 'small_batch', 'small_batch_best_combo', 'strong_weight_decay', 'tuned_neg_multiplier', 'two_layer_gelu', 'wider_hidden']


## Overall Metrics Comparison
Accuracy, Precision, Recall, F1 across all experiments.

In [2]:
names   = sorted(experiments.keys(), key=lambda n: experiments[n]["metrics"]["overall_acc"], reverse=True)
metrics = [experiments[n]["metrics"] for n in names]

fig = go.Figure()
for metric_key, label in [("overall_acc", "Accuracy"), ("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]:
    fig.add_bar(name=label, x=names, y=[m[metric_key] for m in metrics])

fig.update_layout(
    barmode="group",
    title="Overall Metrics by Experiment (sorted by Accuracy)",
    xaxis_title="Experiment",
    yaxis=dict(title="Score", range=[0, 1]),
    legend_title="Metric",
    height=450,
)
fig.show()


## Confusion Matrix Values\nTP, FP, FN, TN across experiments.

In [3]:
fig = go.Figure()
for key, label in [("tp", "TP"), ("fp", "FP"), ("fn", "FN"), ("tn", "TN")]:
    fig.add_bar(name=label, x=names, y=[m[key] for m in metrics])

fig.update_layout(
    barmode="group",
    title="Confusion Matrix Values by Experiment",
    xaxis_title="Experiment",
    yaxis_title="Count",
    legend_title="",
    height=400,
)
fig.show()


## Training & Test Loss Curves
One line per experiment.

In [4]:
import plotly.express as px

fig = make_subplots(rows=2, cols=1, subplot_titles=["Train Loss", "Test Loss"])

exp_names = [n for n, e in experiments.items() if "losses" in e]
palette   = px.colors.qualitative.Plotly
colors    = {name: palette[i % len(palette)] for i, name in enumerate(exp_names)}

for name in exp_names:
    exp    = experiments[name]
    epochs = list(range(1, len(exp["losses"]["train_losses"]) + 1))
    color  = colors[name]
    fig.add_scatter(x=epochs, y=exp["losses"]["train_losses"], name=name, legendgroup=name,
                    mode="lines", line=dict(color=color), row=1, col=1)
    fig.add_scatter(x=epochs, y=exp["losses"]["test_losses"], name=name, legendgroup=name,
                    mode="lines", line=dict(color=color), showlegend=False, row=2, col=1)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Loss")
fig.update_layout(title="Loss Curves by Experiment", height=700)
fig.show()


## Per-Class Accuracy Heatmap
Rows = experiments, columns = classes.

In [5]:
all_classes = sorted({cls for exp in experiments.values() for cls in exp["metrics"]["per_class"]})

z    = []
text = []
for name, exp in experiments.items():
    row      = [exp["metrics"]["per_class"].get(cls, {}).get("accuracy", None) for cls in all_classes]
    text_row = [f"{v:.3f}" if v is not None else "N/A" for v in row]
    z.append(row)
    text.append(text_row)

fig = go.Figure(go.Heatmap(
    z=z, x=all_classes, y=names,
    text=text, texttemplate="%{text}",
    colorscale="RdYlGn", zmin=0, zmax=1,
    colorbar=dict(title="Accuracy"),
))
fig.update_layout(
    title="Per-Class Accuracy by Experiment",
    xaxis_title="Class",
    yaxis_title="Experiment",
    height=80 + 50 * len(experiments),
)
fig.show()


## Protein Coverage Comparison
Fraction of the protein covered by at least one predicted binder.

In [7]:
protein_names = [n for n, e in experiments.items() if "protein" in e]
coverage_pcts = [experiments[n]["protein"]["coverage_pct"] for n in protein_names]
pct_positives = [experiments[n]["protein"]["pct_positive"] for n in protein_names]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Protein Coverage (%)", "% Predicted Positive"])

fig.add_bar(x=protein_names, y=coverage_pcts, name="Coverage %", row=1, col=1)
fig.add_bar(x=protein_names, y=pct_positives, name="% Positive",  row=1, col=2)

fig.update_yaxes(title_text="%")
fig.update_layout(title="Protein Evaluation by Experiment", showlegend=False, height=400)
fig.show()
